In [ ]:
import os, subprocess
os.makedirs("build", exist_ok=True)

In [ ]:
# --- WaveDrom / VCD rendering utilities (auto-generated — do not edit) ---
import json, re, subprocess
from IPython.display import HTML, display

def _vcd_to_wavedrom(vcd_path, max_cycles=80, signals=None):
    """
    Minimal VCD → WaveDrom JSON converter.
    Works for single-bit and multi-bit signals from iverilog output.
    """
    with open(vcd_path) as f:
        vcd = f.read()

    # Parse variable declarations
    var_map = {}  # id_code → (name, width)
    for m in re.finditer(
        r"\$var\s+\w+\s+(\d+)\s+(\S+)\s+(\S+)", vcd
    ):
        width, code, name = int(m.group(1)), m.group(2), m.group(3)
        if signals is None or name in signals:
            var_map[code] = (name, width)

    if not var_map:
        print(f"⚠  No matching signals in {vcd_path}")
        return None

    # Parse value changes
    changes = {}  # id_code → [(time, value), ...]
    current_time = 0
    for line in vcd.splitlines():
        line = line.strip()
        if line.startswith("#"):
            current_time = int(line[1:])
        elif line.startswith("b"):
            parts = line.split()
            if len(parts) == 2 and parts[1] in var_map:
                val = int(parts[0][1:], 2) if parts[0][1:] else 0
                changes.setdefault(parts[1], []).append((current_time, val))
        elif len(line) >= 2 and line[0] in "01xXzZ" and line[1:] in var_map:
            code = line[1:]
            val = {"0": 0, "1": 1, "x": "x", "X": "x", "z": "z", "Z": "z"}[line[0]]
            changes.setdefault(code, []).append((current_time, val))

    # Determine time step (smallest non-zero delta)
    all_times = sorted({t for ch in changes.values() for t, _ in ch})
    if len(all_times) < 2:
        return None
    deltas = [all_times[i+1] - all_times[i] for i in range(len(all_times)-1) if all_times[i+1] != all_times[i]]
    if not deltas:
        return None
    step = min(deltas)
    end_time = min(all_times[-1], step * max_cycles) if max_cycles else all_times[-1]
    sample_times = list(range(0, end_time + 1, step))[:max_cycles]

    # Build WaveDrom signal list
    wave_signals = []
    for code, (name, width) in sorted(var_map.items(), key=lambda x: x[1][0]):
        ch = sorted(changes.get(code, []))
        # Sample at each time point
        samples = []
        cur_val = 0
        ci = 0
        for t in sample_times:
            while ci < len(ch) and ch[ci][0] <= t:
                cur_val = ch[ci][1]
                ci += 1
            samples.append(cur_val)

        if width == 1:
            # Single-bit: WaveDrom wave string
            wave_str = ""
            for v in samples:
                if v == "x":
                    wave_str += "x"
                elif v == "z":
                    wave_str += "z"
                else:
                    wave_str += str(int(v))
            wave_signals.append({"name": name, "wave": wave_str})
        else:
            # Multi-bit: use '=' for data with labels
            wave_str = ""
            data = []
            prev = None
            for v in samples:
                if v == prev:
                    wave_str += "."
                else:
                    wave_str += "="
                    data.append(f"0x{v:X}" if isinstance(v, int) else str(v))
                    prev = v
            wave_signals.append({"name": name, "wave": wave_str, "data": data})

    return {"signal": wave_signals, "config": {"hscale": 1}}


def show_waves(vcd_path="dump.vcd", max_cycles=80, signals=None, width=900):
    """Render VCD waveforms inline using WaveDrom."""
    wd = _vcd_to_wavedrom(vcd_path, max_cycles=max_cycles, signals=signals)
    if wd is None:
        print("No waveform data to display.")
        return
    wd_json = json.dumps(wd)
    html = f"""
    <div id="wd_{id(wd):x}"></div>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/wavedrom.min.js"></script>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/skins/default.js"></script>
    <script>
    (function() {{
        var container = document.getElementById("wd_{id(wd):x}");
        container.innerHTML = '<div id="wd_tgt_{id(wd):x}"><script type="WaveDrom">' +
            JSON.stringify({wd_json}) + '</' + 'script></div>';
        WaveDrom.ProcessAll();
    }})();
    </script>
    """
    display(HTML(html))


def show_wavedrom(wave_dict, width=900):
    """Render a raw WaveDrom JSON dict inline."""
    wd_json = json.dumps(wave_dict)
    html = f"""
    <div id="wd_{id(wave_dict):x}"></div>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/wavedrom.min.js"></script>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/skins/default.js"></script>
    <script>
    (function() {{
        var container = document.getElementById("wd_{id(wave_dict):x}");
        container.innerHTML = '<div id="wd_tgt_{id(wave_dict):x}"><script type="WaveDrom">' +
            JSON.stringify({wd_json}) + '</' + 'script></div>';
        WaveDrom.ProcessAll();
    }})();
    </script>
    """
    display(HTML(html))

print("✓ WaveDrom helpers loaded — use show_waves('dump.vcd') after simulation")

# Day 13 Lab: SystemVerilog for Design

## Overview
Refactor existing Verilog modules into SystemVerilog using `logic`,
`always_ff`, `always_comb`, `enum`, and `struct`. Compare behavior
and synthesis results.

## Prerequisites
- Pre-class video on SystemVerilog design constructs
- Working ALU, traffic light FSM, and UART TX from prior labs

## Toolchain Notes
- Simulation: `iverilog -g2012` (limited SV support)
- Synthesis: `yosys read_verilog -sv`
- Linting: `verilator --lint-only -Wall` (if installed)

## Exercises

| # | Exercise | Time | Key SLOs |
|---|----------|------|----------|
| 1 | ALU Refactor | 25 min | 13.1, 13.2, 13.6 |
| 2 | FSM Refactor | 25 min | 13.2, 13.3, 13.6 |
| 3 | UART TX Refactor | 30 min | 13.1–13.4 |
| 4 | Final Project Design | 30 min | — |
| 5 | Package (Stretch) | 15 min | 13.5 |

## Deliverables
1. SV-refactored ALU passing all original tests
2. SV-refactored FSM with enum states and .name() debug output
3. SV-refactored UART TX with struct-based signal grouping
4. Final project block diagram and module inventory

---
## Exercise Files

The starter files for each exercise are below. Edit the code, then run the simulation/build cells.

#### 📝 Exercise 1 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile alu_sv.sv
// =============================================================================
// alu_sv.sv — ALU Refactored with SystemVerilog
// Day 13, Exercise 1
// =============================================================================
// Refactor the Day 3 ALU to use: logic, always_comb, enum opcodes

module alu_sv #(
    parameter int WIDTH = 4
)(
    input  logic [1:0]       i_opcode,
    input  logic [WIDTH-1:0] i_a,
    input  logic [WIDTH-1:0] i_b,
    output logic [WIDTH-1:0] o_result,
    output logic             o_carry,
    output logic             o_zero
);

    // TODO: Define an enum for opcodes
    // typedef enum logic [1:0] {
    //     OP_ADD = 2'b00,
    //     OP_SUB = 2'b01,
    //     OP_AND = 2'b10,
    //     OP_OR  = 2'b11
    // } alu_op_t;

    // ---- YOUR CODE HERE ----

    // TODO: Replace always @(*) with always_comb
    //   - Set defaults for o_carry and o_result at the top
    //   - Use the enum names in the case statement
    //   - Benefit: if you miss a case, the compiler ERRORS
    //     (unlike Verilog which silently infers a latch)

    // ---- YOUR CODE HERE ----

    assign o_zero = (o_result == '0);  // Note: '0 is SV shorthand for all-zeros

endmodule

In [ ]:
%%writefile Makefile
# Makefile — alu_sv
PROJECT  = alu_sv
TOP      = alu_sv
PCF      = ../go_board.pcf
SRCS     = alu_sv.sv
TB       = tb_alu_sv.sv

DEVICE   = hx1k
PACKAGE  = vq100

IVERILOG_FLAGS = -g2012 -Wall

all: sim

sim: $(TB) $(SRCS)
	iverilog $(IVERILOG_FLAGS) -o sim.vvp $(TB) $(filter-out $(TB),$(SRCS))
	vvp sim.vvp
	@echo "=== Simulation complete ==="

wave: sim
	gtkwave *.vcd &

clean:
	rm -f *.json *.asc *.bin *.vvp *.vcd *.log

.PHONY: all prog sim wave stat clean

#### 📝 Exercise 2 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile traffic_light_sv.sv
// =============================================================================
// traffic_light_sv.sv — Traffic Light FSM Refactored
// Day 13, Exercise 2
// =============================================================================
// Refactor the Day 7 traffic light using: logic, always_ff,
// always_comb, typedef enum

module traffic_light_sv (
    input  logic       i_clk,
    input  logic       i_reset,
    output logic [2:0] o_light   // {red, yellow, green}
);

    // TODO: Replace localparam states with typedef enum
    // typedef enum logic [1:0] {
    //     S_GREEN  = 2'b00,
    //     S_YELLOW = 2'b01,
    //     S_RED    = 2'b10
    // } state_t;
    //
    // state_t state, next_state;

    // ---- YOUR CODE HERE ----

    // Timer (simplified for testing)
    logic [3:0] timer;
    logic       timer_done;
    assign timer_done = (timer == 4'd9);

    // TODO: Block 1 — State register with always_ff
    // ---- YOUR CODE HERE ----

    // TODO: Block 2 — Next-state logic with always_comb
    // ---- YOUR CODE HERE ----

    // TODO: Block 3 — Output logic with always_comb
    // ---- YOUR CODE HERE ----

    // Timer counter
    always_ff @(posedge i_clk) begin
        if (i_reset || timer_done)
            timer <= '0;
        else
            timer <= timer + 1;
    end

endmodule

In [ ]:
%%writefile Makefile
# Makefile — traffic_light_sv
PROJECT  = traffic_light_sv
TOP      = traffic_light_sv
PCF      = ../go_board.pcf
SRCS     = traffic_light_sv.sv
TB       = tb_traffic_light_sv.sv

DEVICE   = hx1k
PACKAGE  = vq100

IVERILOG_FLAGS = -g2012 -Wall

all: sim

sim: $(TB) $(SRCS)
	iverilog $(IVERILOG_FLAGS) -o sim.vvp $(TB) $(filter-out $(TB),$(SRCS))
	vvp sim.vvp
	@echo "=== Simulation complete ==="

wave: sim
	gtkwave *.vcd &

clean:
	rm -f *.json *.asc *.bin *.vvp *.vcd *.log

.PHONY: all prog sim wave stat clean

#### 📝 Exercise 3 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile uart_tx_sv.sv
// =============================================================================
// uart_tx_sv.sv — UART TX Refactored with SystemVerilog
// Day 13, Exercise 3
// =============================================================================
// Refactor the Day 11 UART TX using all SV features:
//   logic, always_ff, enum states, '0 shorthand, parameter int

module uart_tx_sv #(
    parameter int CLK_FREQ  = 25_000_000,
    parameter int BAUD_RATE = 115_200
)(
    input  logic       i_clk,
    input  logic       i_reset,
    input  logic       i_valid,
    input  logic [7:0] i_data,
    output logic       o_tx,
    output logic       o_busy
);

    localparam int CLKS_PER_BIT = CLK_FREQ / BAUD_RATE;
    localparam int BAUD_CNT_W   = $clog2(CLKS_PER_BIT);

    // TODO: Replace localparam states with typedef enum
    // typedef enum logic [1:0] { ... } uart_state_t;
    // uart_state_t state;

    // ---- YOUR ENUM HERE ----

    logic [BAUD_CNT_W-1:0] baud_cnt;
    logic [7:0]            shift_reg;
    logic [2:0]            bit_idx;

    // TODO: Replace wire with logic, use '0 for zero constants
    // logic baud_tick;
    // assign baud_tick = (baud_cnt == BAUD_CNT_W'(CLKS_PER_BIT - 1));

    // ---- YOUR CODE HERE ----

    // TODO: Convert the main always block to always_ff
    //   The FSM logic is the same as Day 11, just with SV syntax:
    //   - Use enum state names
    //   - Use '0 for zero initialization
    //   - Use always_ff @(posedge i_clk)

    // ---- YOUR FSM HERE ----

endmodule

In [ ]:
%%writefile Makefile
# Makefile — uart_tx_sv
PROJECT  = uart_tx_sv
TOP      = uart_tx_sv
PCF      = ../go_board.pcf
SRCS     = uart_tx_sv.sv
TB       = tb_uart_tx_sv.sv

DEVICE   = hx1k
PACKAGE  = vq100

IVERILOG_FLAGS = -g2012 -Wall

all: sim

sim: $(TB) $(SRCS)
	iverilog $(IVERILOG_FLAGS) -o sim.vvp $(TB) $(filter-out $(TB),$(SRCS))
	vvp sim.vvp
	@echo "=== Simulation complete ==="

wave: sim
	gtkwave *.vcd &

clean:
	rm -f *.json *.asc *.bin *.vvp *.vcd *.log

.PHONY: all prog sim wave stat clean